In [1]:
import pandas as pd
import numpy as np
import os

# Load all three datasets
df_ndvi = pd.read_csv('C:/Users/ELITE/Documents/AGROALERT/data_raw/ndvi_raw.csv')
df_weather = pd.read_csv('C:/Users/ELITE/Documents/AGROALERT/data_raw/weather_raw.csv')
df_lst = pd.read_csv('C:/Users/ELITE/Documents/AGROALERT/data_raw/soil_moisture_raw.csv')

# Convert dates
df_ndvi['date'] = pd.to_datetime(df_ndvi['date'])
df_weather['date'] = pd.to_datetime(df_weather['date'])
df_lst['date'] = pd.to_datetime(df_lst['date'])

print("NDVI shape:", df_ndvi.shape)
print("Weather shape:", df_weather.shape)
print("LST shape:", df_lst.shape)
print("\nNDVI sample:")
print(df_ndvi.head(3))
print("\nWeather sample:")
print(df_weather.head(3))
print("\nLST sample:")
print(df_lst.head(3))

NDVI shape: (247, 4)
Weather shape: (3650, 8)
LST shape: (367, 4)

NDVI sample:
    community      region       date      ndvi
0  Bolgatanga  Upper East 2022-01-01  0.208887
1  Bolgatanga  Upper East 2022-01-01  0.206289
2  Bolgatanga  Upper East 2022-01-06  0.185902

Weather sample:
  community    region       date  rainfall_mm  temp_max  temp_min  humidity  \
0    Tamale  Northern 2022-01-01          0.0      34.5      19.7        30   
1    Tamale  Northern 2022-01-02          0.0      34.8      21.7        24   
2    Tamale  Northern 2022-01-03          0.2      30.1      23.3        31   

    et0  
0  6.62  
1  6.84  
2  5.74  

LST sample:
    community      region       date  lst_celsius
0  Bolgatanga  Upper East 2022-01-01    35.620115
1  Bolgatanga  Upper East 2022-01-09    37.958092
2  Bolgatanga  Upper East 2022-01-17    35.993902


In [2]:
# Resample NDVI to weekly (it has multiple readings per week)
df_ndvi_weekly = (df_ndvi.groupby(['community', 'region', 
                  pd.Grouper(key='date', freq='W')])
                  ['ndvi'].mean().reset_index())

# Resample weather to weekly
df_weather_weekly = (df_weather.groupby(['community', 'region',
                     pd.Grouper(key='date', freq='W')])
                     .agg({
                         'rainfall_mm': 'sum',
                         'temp_max': 'mean',
                         'temp_min': 'mean',
                         'humidity': 'mean',
                         'et0': 'sum'
                     }).reset_index())

# Resample LST to weekly
df_lst_weekly = (df_lst.groupby(['community', 'region',
                pd.Grouper(key='date', freq='W')])
                ['lst_celsius'].mean().reset_index())

# Merge all three on community + date
df_master = df_ndvi_weekly.merge(
    df_weather_weekly, on=['community', 'region', 'date'], how='outer')
df_master = df_master.merge(
    df_lst_weekly, on=['community', 'region', 'date'], how='outer')

df_master = df_master.sort_values(['community', 'date']).reset_index(drop=True)

print(f"Master dataset shape: {df_master.shape}")
print(f"\nColumns: {list(df_master.columns)}")
print(f"\nMissing values:\n{df_master.isnull().sum()}")
print(f"\nSample:")
print(df_master.head(10))

Master dataset shape: (525, 10)

Columns: ['community', 'region', 'date', 'ndvi', 'rainfall_mm', 'temp_max', 'temp_min', 'humidity', 'et0', 'lst_celsius']

Missing values:
community        0
region           0
date             0
ndvi           357
rainfall_mm      0
temp_max         0
temp_min         0
humidity         0
et0              0
lst_celsius    163
dtype: int64

Sample:
    community      region       date      ndvi  rainfall_mm   temp_max  \
0  Bolgatanga  Upper East 2022-01-02  0.207588          0.0  34.950000   
1  Bolgatanga  Upper East 2022-01-09  0.184937          0.0  33.500000   
2  Bolgatanga  Upper East 2022-01-16  0.189436          0.0  35.342857   
3  Bolgatanga  Upper East 2022-01-23  0.184509          0.0  32.900000   
4  Bolgatanga  Upper East 2022-01-30  0.198099          0.0  34.685714   
5  Bolgatanga  Upper East 2022-02-06  0.183988          0.0  34.671429   
6  Bolgatanga  Upper East 2022-02-13  0.192174          0.0  37.314286   
7  Bolgatanga  Upper Eas

In [3]:
# Fill missing values
df_master = df_master.sort_values(['community', 'date'])

# Fill NDVI and LST gaps using interpolation per community
for community in df_master['community'].unique():
    mask = df_master['community'] == community
    df_master.loc[mask, 'ndvi'] = (df_master.loc[mask, 'ndvi']
                                   .interpolate(method='linear')
                                   .fillna(method='bfill')
                                   .fillna(method='ffill'))
    df_master.loc[mask, 'lst_celsius'] = (df_master.loc[mask, 'lst_celsius']
                                          .interpolate(method='linear')
                                          .fillna(method='bfill')
                                          .fillna(method='ffill'))

# Add derived features
# 1. NDVI anomaly (current vs 4-week rolling average)
df_master['ndvi_anomaly'] = (df_master.groupby('community')['ndvi']
                             .transform(lambda x: x - x.rolling(4, min_periods=1).mean()))

# 2. Rainfall deficit (current vs 4-week rolling average)
df_master['rainfall_deficit'] = (df_master.groupby('community')['rainfall_mm']
                                 .transform(lambda x: x - x.rolling(4, min_periods=1).mean()))

# 3. SPEI proxy (standardized rainfall - ET0 over 4 weeks)
df_master['water_balance'] = df_master['rainfall_mm'] - df_master['et0']
df_master['spei_proxy'] = (df_master.groupby('community')['water_balance']
                           .transform(lambda x: 
                               (x.rolling(4, min_periods=1).sum() - 
                                x.rolling(4, min_periods=1).sum().mean()) / 
                               (x.rolling(4, min_periods=1).sum().std() + 1e-8)))

# 4. Drought label
# 1 = drought stress, 0 = no drought
df_master['drought_label'] = (
    (df_master['spei_proxy'] < -1.0) & 
    (df_master['ndvi_anomaly'] < -0.05)
).astype(int)

print(f"Final dataset shape: {df_master.shape}")
print(f"\nMissing values after filling:\n{df_master.isnull().sum()}")
print(f"\nDrought label distribution:")
print(df_master['drought_label'].value_counts())
print(f"\nSample with all features:")
print(df_master.head(5).to_string())

TypeError: NDFrame.fillna() got an unexpected keyword argument 'method'

In [4]:
# Fill missing values
df_master = df_master.sort_values(['community', 'date'])

for community in df_master['community'].unique():
    mask = df_master['community'] == community
    df_master.loc[mask, 'ndvi'] = (df_master.loc[mask, 'ndvi']
                                   .interpolate(method='linear')
                                   .bfill()
                                   .ffill())
    df_master.loc[mask, 'lst_celsius'] = (df_master.loc[mask, 'lst_celsius']
                                          .interpolate(method='linear')
                                          .bfill()
                                          .ffill())

# Add derived features
df_master['ndvi_anomaly'] = (df_master.groupby('community')['ndvi']
                             .transform(lambda x: x - x.rolling(4, min_periods=1).mean()))

df_master['rainfall_deficit'] = (df_master.groupby('community')['rainfall_mm']
                                 .transform(lambda x: x - x.rolling(4, min_periods=1).mean()))

df_master['water_balance'] = df_master['rainfall_mm'] - df_master['et0']
df_master['spei_proxy'] = (df_master.groupby('community')['water_balance']
                           .transform(lambda x:
                               (x.rolling(4, min_periods=1).sum() -
                                x.rolling(4, min_periods=1).sum().mean()) /
                               (x.rolling(4, min_periods=1).sum().std() + 1e-8)))

df_master['drought_label'] = (
    (df_master['spei_proxy'] < -1.0) &
    (df_master['ndvi_anomaly'] < -0.05)
).astype(int)

print(f"Final dataset shape: {df_master.shape}")
print(f"\nMissing values:\n{df_master.isnull().sum()}")
print(f"\nDrought label distribution:")
print(df_master['drought_label'].value_counts())
print(f"\nSample:")
print(df_master.head(5).to_string())

Final dataset shape: (525, 15)

Missing values:
community           0
region              0
date                0
ndvi                0
rainfall_mm         0
temp_max            0
temp_min            0
humidity            0
et0                 0
lst_celsius         0
ndvi_anomaly        0
rainfall_deficit    0
water_balance       0
spei_proxy          0
drought_label       0
dtype: int64

Drought label distribution:
drought_label
0    515
1     10
Name: count, dtype: int64

Sample:
    community      region       date      ndvi  rainfall_mm   temp_max   temp_min   humidity    et0  lst_celsius  ndvi_anomaly  rainfall_deficit  water_balance  spei_proxy  drought_label
0  Bolgatanga  Upper East 2022-01-02  0.207588          0.0  34.950000  21.500000  18.000000  15.55    35.620115      0.000000               0.0         -15.55    0.640702              0
1  Bolgatanga  Upper East 2022-01-09  0.184937          0.0  33.500000  21.385714  25.000000  48.89    37.958092     -0.011325             

In [5]:
# Save master dataset
os.makedirs('C:/Users/ELITE/Documents/AGROALERT/data_processed', exist_ok=True)
df_master.to_csv('C:/Users/ELITE/Documents/AGROALERT/data_processed/master_dataset.csv', index=False)
print("Master dataset saved!")
print(f"\nFinal columns: {list(df_master.columns)}")
print(f"\nData ranges:")
print(df_master[['ndvi','rainfall_mm','temp_max','lst_celsius',
                 'ndvi_anomaly','spei_proxy','drought_label']].describe().round(3))

Master dataset saved!

Final columns: ['community', 'region', 'date', 'ndvi', 'rainfall_mm', 'temp_max', 'temp_min', 'humidity', 'et0', 'lst_celsius', 'ndvi_anomaly', 'rainfall_deficit', 'water_balance', 'spei_proxy', 'drought_label']

Data ranges:
          ndvi  rainfall_mm  temp_max  lst_celsius  ndvi_anomaly  spei_proxy  \
count  525.000      525.000   525.000      525.000       525.000     525.000   
mean     0.353       25.989    32.367       31.893         0.001      -0.000   
std      0.129       26.446     2.930        3.678         0.028       0.996   
min      0.021        0.000    26.971       23.082        -0.173      -2.089   
25%      0.242        1.100    29.871       29.332        -0.005      -0.791   
50%      0.349       20.700    32.186       31.308         0.003      -0.084   
75%      0.472       41.900    34.586       34.252         0.010       0.841   
max      0.585      149.200    39.957       43.017         0.130       2.702   

       drought_label  
count  

In [6]:
import pandas as pd
import numpy as np
import os

# Load all three scaled datasets
df_ndvi = pd.read_csv('C:/Users/ELITE/Documents/AGROALERT/data_raw/ndvi_raw.csv')
df_weather = pd.read_csv('C:/Users/ELITE/Documents/AGROALERT/data_raw/weather_raw.csv')
df_lst = pd.read_csv('C:/Users/ELITE/Documents/AGROALERT/data_raw/soil_moisture_raw.csv')

df_ndvi['date'] = pd.to_datetime(df_ndvi['date'])
df_weather['date'] = pd.to_datetime(df_weather['date'])
df_lst['date'] = pd.to_datetime(df_lst['date'])

# Resample to weekly
df_ndvi_w = (df_ndvi.groupby(['community','region',
             pd.Grouper(key='date',freq='W')])['ndvi']
             .mean().reset_index())

df_weather_w = (df_weather.groupby(['community','region',
               pd.Grouper(key='date',freq='W')])
               .agg({'rainfall_mm':'sum','temp_max':'mean',
                     'temp_min':'mean','humidity':'mean','et0':'sum'})
               .reset_index())

df_lst_w = (df_lst.groupby(['community','region',
            pd.Grouper(key='date',freq='W')])['lst_celsius']
            .mean().reset_index())

# Merge
df = df_ndvi_w.merge(df_weather_w, on=['community','region','date'], how='outer')
df = df.merge(df_lst_w, on=['community','region','date'], how='outer')
df = df.sort_values(['community','date']).reset_index(drop=True)

# Fill gaps
for community in df['community'].unique():
    mask = df['community'] == community
    df.loc[mask,'ndvi'] = df.loc[mask,'ndvi'].interpolate().bfill().ffill()
    df.loc[mask,'lst_celsius'] = df.loc[mask,'lst_celsius'].interpolate().bfill().ffill()

# Feature engineering
df['ndvi_anomaly'] = (df.groupby('community')['ndvi']
                      .transform(lambda x: x - x.rolling(4,min_periods=1).mean()))
df['rainfall_deficit'] = (df.groupby('community')['rainfall_mm']
                          .transform(lambda x: x - x.rolling(4,min_periods=1).mean()))
df['water_balance'] = df['rainfall_mm'] - df['et0']
df['spei_proxy'] = (df.groupby('community')['water_balance']
                    .transform(lambda x:
                        (x.rolling(4,min_periods=1).sum() -
                         x.rolling(4,min_periods=1).sum().mean()) /
                        (x.rolling(4,min_periods=1).sum().std() + 1e-8)))

# Drought label
df['drought_label'] = (
    (df['spei_proxy'] < -1.0) &
    (df['ndvi_anomaly'] < -0.05)
).astype(int)

# Save
os.makedirs('C:/Users/ELITE/Documents/AGROALERT/data_processed', exist_ok=True)
df.to_csv('C:/Users/ELITE/Documents/AGROALERT/data_processed/master_dataset.csv', index=False)

print(f"Master dataset shape: {df.shape}")
print(f"Communities: {df['community'].nunique()}")
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"\nDrought label distribution:")
print(df['drought_label'].value_counts())
print(f"\nDrought cases per community:")
print(df.groupby('community')['drought_label'].sum().sort_values(ascending=False))

Master dataset shape: (1575, 15)
Communities: 15
Missing values: 0

Drought label distribution:
drought_label
0    1547
1      28
Name: count, dtype: int64

Drought cases per community:
community
Ho                  5
Damongo             4
Goaso               4
Koforidua           4
Sunyani             3
Techiman            2
Bolgatanga          1
Tamale              1
Dambai              1
Kumasi              1
Nalerigu            1
Sekondi-Takoradi    1
Cape Coast          0
Sefwi Wiawso        0
Wa                  0
Name: drought_label, dtype: int64
